In [579]:
from strands import Agent, tool

In [580]:
LBS_TO_KG = 0.453592
INCH_TO_CM = 2.54


def lbs_to_kg(weight_lbs: float) -> float:
    """Convert weight from pounds to kilograms."""
    return weight_lbs * LBS_TO_KG


def feet_inches_to_cm(feet: int, inches: int) -> float:
    """Convert height from feet/inches to centimeters."""
    total_inches = (feet * 12) + inches
    return total_inches * INCH_TO_CM

In [581]:
import json
import re

def extract_json(text):
    """Extract a JSON object or array from an LLM response."""

    text = text.strip()

    # Remove Markdown code fences if present
    text = re.sub(r"^```(?:json)?", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)

    if match:
        return json.loads(match.group(1))

    raise ValueError("No valid JSON found.")

In [582]:
def calculate_nutrition_targets(user_input):
    weight_lbs = user_input["weight_lbs"]
    age = user_input["age"]
    sex = user_input["sex"]
    activity = user_input["activity_level"]
    goal = user_input["goal"]

    weight_kg = lbs_to_kg(weight_lbs)
    height_cm = feet_inches_to_cm(
        user_input["height_feet"],
        user_input["height_inches"]
    )

    if sex.lower() == "male":
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    else:
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161

    activity_levels = {
        "sedentary": 1.2,
        "light": 1.375,
        "moderate": 1.55,
        "active": 1.725,
        "very active": 1.9,
    }

    tdee = bmr * activity_levels[activity]

    if goal.lower() == "fat loss":
        calorie_goal = tdee * 0.80
    elif goal.lower() == "muscle gain":
        calorie_goal = tdee * 1.10
    else:
        calorie_goal = tdee

    protein_goal = weight_lbs

    return {
        "bmr": round(bmr),
        "tdee": round(tdee),
        "calorie_goal": round(calorie_goal),
        "protein_goal": round(protein_goal),
    }

In [583]:
# Recipe sources allowed for nutrition searches
TRUSTED_DOMAINS = {
    "allrecipes.com",
    "eatingwell.com",
    "skinnytaste.com",
    "foodnetwork.com",
    "delish.com",
    "bbcgoodfood.com",
}

In [584]:
def is_recipe_page(url: str, title: str) -> bool:
    """
    Filter out collection pages and keep individual recipes.
    """

    blocked_terms = [
        "ideas",
        "best",
        "top",
        "collection",
        "roundup",
        "list",
        "high-protein",
        "meal-plan"
    ]

    url_lower = url.lower()
    title_lower = title.lower()

    # reject obvious collection pages
    for term in blocked_terms:
        if term in url_lower or term in title_lower:
            return False

    return True

In [585]:
from typing import List, Dict, Any
import requests
from bs4 import BeautifulSoup

def search_recipes_web(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    """Search trusted recipe websites for recipes matching a query."""

    url = "https://html.duckduckgo.com/html/"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.post(
            url,
            data={"q": query},
            headers=headers,
            timeout=10
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        results = []
        
        for item in soup.find_all("div", class_="result")[:max_results]:
            link = item.find("a", class_="result__a")

            if not link:
                continue

            recipe_url = link.get("href", "")
            title = link.get_text(strip=True)

            snippet = item.find("a", class_="result__snippet")
            snippet_text = snippet.get_text(strip=True) if snippet else ""
            if any(domain in recipe_url.lower() for domain in TRUSTED_DOMAINS):
                print("FOUND:", title)
                print(recipe_url)
                results.append({
                    "title": title,
                    "url": recipe_url,
                    "snippet": snippet_text
                })

        return results

    except requests.RequestException:
        return []

In [586]:
def collect_recipe_results(calorie_target, ingredients):

    searches = [
        f"{ingredients} breakfast recipe",
        f"chicken rice broccoli recipe",
        f"high protein chicken recipe",
        f"protein snack recipe"
    ]


    results = []

    for query in searches:
        print("Searching:", query)
        results.extend(search_recipes_web(query))

    return results

In [599]:
@tool
def recipe_search_tool(
    calorie_target: int,
    ingredients: str
):

    """
    Searches the web for high-protein recipes.
    """

    results =[]
    
    results = collect_recipe_results(
        calorie_target=calorie_target,
        ingredients=ingredients
    )


    return results

In [588]:
# Agent 1: Recipe Finder Agent

recipe_agent = Agent(
    name="RecipeResearchAgent",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""
You are a recipe generation and retrieval specialist.

Your task is to create structured recipe candidates using:
- web search results
- user ingredients
- nutrition targets

Use web results as inspiration.

If nutrition information is unavailable:
- generate realistic nutrition estimates
- create recipes using available ingredients

Do not create a meal plan.
Only create individual recipe candidates.

Inputs:
- calorie target
- protein target
- available ingredients
- recipe search results

Select recipes that:
- are high protein (25-50g per serving)
- fit the calorie target
- use available ingredients when possible
- come from reliable sources

Do NOT:
- create a meal plan
- modify recipes
- Nutrition values must be realistic estimates based on standard serving sizes.

Exclude recipes if required nutrition data is missing.

If a result is a collection page:
- Extract individual recipes mentioned in the snippet only if a URL is available.
- Otherwise ignore it.

If no recipes contain complete nutrition information:
return []

Never generate recipes from ingredients.
Never estimate calories.

Return ONLY valid JSON.

Each recipe must follow:

{
"name": string,
"meal_type": "breakfast | lunch | dinner | snack",
"calories": number,
"protein_g": number,
"carbs_g": number,
"fat_g": number,
"ingredients": [],
"url": string
}

Output:

[
  {
    "name": "",
    "meal_type": "",
    "calories": 0,
    "protein_g": 0,
    "carbs_g": 0,
    "fat_g": 0,
    "ingredients": [],
    "url": ""
  }
]
"""
)

In [598]:
@tool
def recipe_finder_tool(
    calorie_target: int,
    protein_target: int,
    ingredients: str,
):
    
    """
    Finds high-protein recipes and generates meal options
    based on user nutrition targets.
    """

    search_results = recipe_search_tool(
        calorie_target=calorie_target,
        ingredients=ingredients,
    )

    prompt = f"""
You are a recipe extraction agent.

Nutrition targets:
Calories: {calorie_target}
Protein: {protein_target}g

Available ingredients:
{ingredients}

Search results:
{json.dumps(search_results, indent=2)}

Create realistic high-protein recipes.

Rules:
- Use the provided ingredients when possible.
- Recipes must have realistic nutrition estimates.
- Include breakfast, lunch, dinner, and snack options.
- Do not explain anything.
- Return ONLY valid JSON.

Format:

[
    {{
        "name": "",
        "meal_type": "",
        "calories": 0,
        "protein_g": 0,
        "carbs_g": 0,
        "fat_g": 0,
        "ingredients": []
    }}
]
"""

    response = recipe_agent(prompt)

    response_text = response.message["content"][0]["text"]

    return extract_json(response_text)

In [ ]:
@tool
def validate_meal_plan(
    meals: list,
    calorie_target: int,
    protein_target: int
):
    """
    Validate a generated meal plan against nutrition targets.

    Rules:
    - Calories must be within +/- 10% of target
    - Protein must meet or exceed target

    Returns validation status and feedback.
    """

    total_calories = sum(
        meal["calories"] for meal in meals
    )

    total_protein = sum(
        meal["protein_g"] for meal in meals
    )

    calorie_difference = total_calories - calorie_target
    protein_difference = total_protein - protein_target

    calories_valid = (
        abs(calorie_difference)
        <= calorie_target * 0.10
    )

    protein_valid = (
        total_protein >= protein_target
    )

    is_valid = calories_valid and protein_valid


    feedback = ""

    if not calories_valid:
        feedback += (
            f"Calories are {abs(calorie_difference)} kcal "
            f"{'under' if calorie_difference < 0 else 'over'} target. "
        )

    if not protein_valid:
        feedback += (
            f"Protein is {abs(protein_difference)}g below target."
        )


    return {
    "is_valid": is_valid,
    "feedback": feedback.strip(),
    "daily_totals": {
        "calories": total_calories,
        "protein_g": total_protein
    },
    "meals": meals
}

In [592]:
# Agent 2: Meal Plan Optimizer Agent

optimizer_agent = Agent(
    name="MealPlanningAgent",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""
You are a meal planning optimizer.

Your task is to create a daily meal plan using provided recipes.

Inputs:
- User calorie target
- User protein target
- Available recipes

Goals:
1. Select breakfast, lunch, and dinner.
2. Add a snack only if needed.
3. Match calorie target within 10%.
4. Match protein target within 90%-110%.
5. Prioritize balanced meals over maximum protein.

Meal rules:
- Exactly one breakfast.
- Exactly one lunch.
- Exactly one dinner.
- Do not duplicate meal types.
- Prefer recipes using user ingredients.

If calories are too low:
Increase portions or add realistic sides:
- rice
- oats
- fruit
- peanut butter
- avocado
- olive oil

Do not fix calorie shortages by adding unnecessary protein-heavy foods.

Before returning:
1. Calculate total calories and protein.
2. Call validate_meal_plan.
3. If validation fails, adjust the plan and retry.
4. Return only the final valid JSON.

Output format:

{
  "meals": [
    {
      "name": "",
      "meal_type": "",
      "calories": 0,
      "protein_g": 0,
      "carbs_g": 0,
      "fat_g": 0,
      "ingredients": []
    }
  ],
  "daily_total": {
    "calories": 0,
    "protein_g": 0
  }
}

Rules:
- Output ONLY JSON.
- No markdown.
- No explanations.
""",
    tools=[validate_meal_plan]
)

In [ ]:
@tool
def meal_optimizer_tool(recipes, calorie_target, protein_target, feedback=""):
    
    """
    Generate a meal plan from available recipes.
    """

    feedback_section = ""

    if feedback:
        feedback_section = f"""
Previous validation feedback:
{feedback}

Revise the meal plan to satisfy this feedback.
"""

    optimizer_prompt = f"""
You are a nutrition planner.

User nutrition targets:
Calories: {calorie_target} kcal
Protein: {protein_target} g

Available recipes:
{json.dumps(recipes, indent=2)}

{feedback_section}

Requirements:

Meal structure:
- Include breakfast, lunch, and dinner.
- Add snacks only if necessary.
- Prioritize realistic meals a person would actually eat.

Nutrition:
- Calories must be within 10% of target.
- Protein should meet the target without excessive overages.
- Prioritize calorie accuracy over maximizing protein.

If calories are low:
Use calorie-dense additions:
- rice
- oats
- avocado
- olive oil
- peanut butter
- fruit

Do not:
- invent unrealistic nutrition values
- replace ingredients unnecessarily
- add excessive protein sources

Return ONLY valid JSON.

Schema:

{
 "meals": [
   {
    "name": "",
    "meal_type": "",
    "calories": 0,
    "protein_g": 0,
    "carbs_g": 0,
    "fat_g": 0,
    "ingredients": []
   }
 ],
 "daily_total": {
    "calories": 0,
    "protein_g": 0
 }
}
"""


    meal_plan_result = optimizer_agent(optimizer_prompt)

    meal_plan_text = meal_plan_result.message["content"][0]["text"]

    return extract_json(meal_plan_text)

In [600]:
def print_meal_plan(plan):

    print("""
==================================================
        CUTBUDDY AI NUTRITION ASSISTANT
==================================================

🤖 ORCHESTRATOR AGENT
Coordinating nutrition planning workflow...

Agents:
✓ Nutrition Calculator Agent
✓ Recipe Finder Agent
✓ Meal Optimization Agent

==================================================
        PERSONALIZED NUTRITION TARGETS
==================================================
""")

    print(f"""
Calories Target: {plan['daily_total']['calories']} kcal
Protein Target: {plan['daily_total']['protein_g']}g
""")

    print("""
==================================================
             GENERATED MEAL PLAN
==================================================
""")

    for meal in plan["meals"]:
        print(f"""
{meal['meal_type'].upper()}
-------------------------
{meal['name']}

Calories: {meal['calories']} kcal
Protein: {meal['protein_g']}g
Carbs: {meal['carbs_g']}g
Fat: {meal['fat_g']}g

Ingredients:
""")

        for ingredient in meal["ingredients"]:
            print(f"• {ingredient}")

    print("""
==================================================
SUCCESS
Meal plan generated by multi-agent AI workflow.
==================================================
""")

In [602]:
orchestrator_agent = Agent(
    name="MealPlanningOrchestrator",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[
        recipe_finder_tool,
        meal_optimizer_tool,
        validate_meal_plan,
    ],
    system_prompt="""
You are a workflow orchestration agent.

Your responsibility is coordinating specialized nutrition tools.

Available components:

1. Recipe Finder Agent
   - Searches and extracts high-protein recipes.

2. Meal Optimization Agent
   - Generates a personalized daily meal plan.

3. Validation Tool
   - Checks whether the meal plan satisfies nutrition targets.


Workflow:

1. Call recipe_finder_tool exactly once.
2. Send recipe results to meal_optimizer_tool.
3. Validate the generated meal plan.

Validation criteria:
- Calories within +/- 10% of target.
- Protein >= target.

If validation succeeds:
- stop immediately.
- return only the final meal plan JSON.

If validation fails:
- send feedback to meal_optimizer_tool.
- regenerate.
- validate again.
- maximum 3 attempts.

Never:
- explain reasoning
- mention tools
- provide progress updates
- return validation metadata

Final response:
Return ONLY valid JSON matching this schema:

{
  "meals": [],
  "daily_total": {}
}
"""
)

In [596]:
def main():
    user_input = {
        "weight_lbs": 154,
        "height_feet": 5,
        "height_inches": 9,
        "age": 20,
        "sex": "male",
        "activity_level": "moderate",
        "ingredients": "chicken, eggs, rice, broccoli",
        "goal": "fat loss",
    }

    targets = calculate_nutrition_targets(user_input)

    print(f"BMR: {targets['bmr']}")
    print(f"TDEE: {targets['tdee']}")
    print(f"Calories: {targets['calorie_goal']}")
    print(f"Protein: {targets['protein_goal']} g")

    prompt = f"""
Generate a personalized daily meal plan.

Nutrition Targets:
- Calories: {round(targets["calorie_goal"])}
- Protein: {round(targets["protein_goal"])} g

Available Ingredients:
{user_input["ingredients"]}

Goal:
{user_input["goal"]}

Coordinate the available tools to produce a validated meal plan.
Return only the final meal plan.
"""

    response = orchestrator_agent(prompt)

    final_plan = response.message["content"][0]["text"]

    meal_plan = extract_json(final_plan)

    print_meal_plan(meal_plan)

In [597]:
main()

BMR: 1699
TDEE: 2633
Calories: 2107
Protein: 154 g

Tool #1: recipe_finder_tool
🍳 Recipe finder tool called
Searching: chicken, eggs, rice, broccoli breakfast recipe
FOUND: Broccoli, Rice, Cheese, and Chicken Casserole Recipe
https://www.allrecipes.com/recipe/25490/broccoli-rice-cheese-and-chicken-casserole/
Searching: chicken rice broccoli recipe
FOUND: Broccoli, Rice, Cheese, and Chicken Casserole Recipe
https://www.allrecipes.com/recipe/25490/broccoli-rice-cheese-and-chicken-casserole/
Searching: high protein chicken recipe
FOUND: 15+ High-Protein Chicken Dinner Recipes - EatingWell
https://www.eatingwell.com/high-protein-chicken-dinner-recipes-11982405
FOUND: 10+ High-Protein 30-Minute Chicken Dinner Recipes - EatingWell
https://www.eatingwell.com/high-protein-30-minute-chicken-dinners-11941472
Searching: protein snack recipe
RESULT COUNT: 4
[{'title': 'Broccoli, Rice, Cheese, and Chicken Casserole Recipe', 'url': 'https://www.allrecipes.com/recipe/25490/broccoli-rice-cheese-and-ch